# Next generation sequencing data and population dynamics for novel GNRA/receptors isolated by in vitro selection and evolution Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.sgrj-01tk/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Here we enumerate all available record sets, their fields, and columns using the unique `@id` for each entity. This allows for precise referencing and extraction in later steps.

In [ ]:
# List available record sets with their IDs and fields
record_sets = dataset.record_sets
print(f"Total record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set Name: {rs.name}\n  @id: {rs.id}")
    print("  Fields:")
    for f in rs.fields:
        print(f"    - {f.name} (@id: {f.id}) | DataType: {f.data_type}")
    print("  Columns:")
    for col in rs.columns:
        print(f"    - {col.name} (@id: {col.id})")
    print("\n")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Entities are referenced by their `@id` throughout.

We'll demonstrate with the first two record sets listed in the previous overview.

In [ ]:
# Extract data from available record sets listed above
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_set_ids[:2]:  # As an example, use first two record sets
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records for Record Set @id: {rs_id}")
    print(f"Columns: {dataframes[rs_id].columns.tolist()}")
    print(dataframes[rs_id].head())

# Select one record set for further analysis
selected_rs_id = record_set_ids[0]  # Pick the first for demonstration

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps on extracted records:
- Filtering records based on a numeric field
- Normalizing values
- Grouping by a categorical attribute

Fields and columns are referenced by their `@id` as listed earlier.

In [ ]:
# Choose numeric and categorical fields for EDA
df = dataframes[selected_rs_id]

# Find numeric and categorical field @id from overview
numeric_field_id = None
group_field_id = None
for rs in dataset.record_sets:
    if rs.id == selected_rs_id:
        for f in rs.fields:
            if f.data_type in ["Float", "Integer", "Number"] and not numeric_field_id:
                numeric_field_id = f.id
            if f.data_type == "Text" and not group_field_id:
                group_field_id = f.id
        break

print(f"Using numeric_field_id: {numeric_field_id}")
print(f"Using group_field_id: {group_field_id}")

if numeric_field_id and numeric_field_id in df.columns:
    threshold = df[numeric_field_id].mean()  # As example, use mean as threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Group and aggregate by group_field if available
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

For example, plot the normalized numeric field or compare group means.

In [ ]:
# Visualization example: distribution and boxplot
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id} in Record Set @id: {selected_rs_id}")
    plt.show()

    # Grouped boxplot
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field_id} distribution by {group_field_id}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Demonstrated loading, overview, and extraction of records using `mlcroissant`.
- Fields and record sets referenced by unique `@id` ensure reproducibility and clarity.
- Performed basic filtering, normalization, and grouping on extracted record data, visualized distributions.
- Dataset contains rich sequence information suitable for population and motif analysis.

Further analysis may include advanced statistical models or sequence clustering methods using the provided fields and columns.